# Test Stage 5b — Known-answer tests on the correction filter

**Run this when Stage 5 fails and you need to know *which part* is at fault.**

Stage 5 measures the filter through a signal, so a failure could be the filter, the
coefficients, the decimator feeding it, or the way the amplitude is measured. This notebook
does something different: it **loads filters whose answer is known in advance** and checks the
hardware gives that answer.

| Test | Filter loaded | Must produce | A failure means |
|---|---|---|---|
| A | every coefficient zero | exactly zero | the accumulator is not cleared between passes, or the output is latched mid-pass |
| B | centre tap = 65536 | half the input | the multiply, the accumulate or the output shift is wrong |
| C | first tap = 65536 | \|cos(308πu)\| — two taps 308 apart | **the pre-add, or the sample history addressing** |
| D | the designed filter | the designed response | the coefficients in the bitstream are not the designed ones |

These tests have already earned their keep once: **Test B passed while Test C railed**, which
placed the fault in the one thing C does and B does not — adding the two mirrored samples
together. That is the whole value of a known answer. See `../CHANGELOG.md`.

---

## Two things that will trip you up if you skip them

**Coefficients are 18-bit signed, so they run −131072 … +131071.** The scaling is 2¹⁷ = 131072,
which means the number for "gain 1" is one larger than the largest that fits. Write 131072 and
it comes back as **−131072**. That is not a fault — it is the top bit, which is the sign bit.
The designed filter never needs it (its largest coefficient is 71142), so this only bites when
you hand-load a test filter. These tests use **65536, which is gain ½**, and predict
accordingly.

**The generator is not locked to the board.** They are two independent crystals, typically a
few parts per million apart — tens of hertz at 10 MHz. So "put the reference tone exactly on
the signal" is never exact: what comes out of the decimator is not a steady number but a slow
rotation, and `PEEK_I` alone sweeps its whole range. Everything below therefore measures
**amplitude**, which does not care, and the setup cell measures the residual beat so you can
see it for yourself.

---

## Setup

A signal generator into **IN1**, a few hundred mV, at a frequency you know.

---

### Simulation or the real trap?

**This stage never touches the emulated coil.** It measures the acquisition chain itself —
the part that listens — using a bench signal generator plugged into IN1. That is already a
real hardware measurement, so nothing here changes when the trap is connected.

One thing *is* worth doing again for real work. These tests run at a convenient
`F_SIG = 10 MHz` and one fixed averaging setting, and droop, noise and timing all depend on
both. **Before taking trap data, re-run this stage at the frequency and averaging you will
actually use** — around 40 MHz for the coil — so the instrument is characterised where it
is used rather than where it was convenient to test.


In [ ]:
import os, mmap, struct, time
import numpy as np

FS_CLK   = 125e6
ACQ_BASE = 0x40600000

R_CTRL, R_LO_FINC, R_DEC_LOG2, R_SRC_SEL = 0x00, 0x04, 0x08, 0x0C
R_PEEK_I, R_PEEK_Q, R_SAMPLE_CNT         = 0x18, 0x1C, 0x20
R_STATUS, R_STATUS2                      = 0x24, 0x38
R_COEF_ADDR, R_COEF_DATA                 = 0x40, 0x44
R_BUILD_ID                               = 0x48

NHALF, NTAPS   = 155, 309
COEFW          = 18            # coefficients are 18-bit signed ...
COEF_FRAC      = 17            # ... scaled by 2^17
COEF_MAX       =  (1 << (COEFW - 1)) - 1     # +131071
COEF_MIN       = -(1 << (COEFW - 1))         # -131072
HALF           = 1 << (COEF_FRAC - 1)        #  65536  ->  gain exactly 0.5
NSTAGE         = 4


def s32(v):
    """Undo a 32-BIT sign. The hardware sign-extends 18 bits to 32 before
    returning a coefficient, so a stored -1 arrives as 0xFFFFFFFF."""
    return v - (1 << 32) if v & 0x80000000 else v


class Acq(object):
    """Every access is a SINGLE 32-bit read or write. A bulk mmap slice becomes
    a burst the bus bridge cannot acknowledge; it locks the bus and reboots the
    board. Do not optimise these loops into slices."""

    def __init__(self, base=ACQ_BASE):
        self.fd = os.open('/dev/mem', os.O_RDWR | os.O_SYNC)
        self.m  = mmap.mmap(self.fd, 0x1000, offset=base)

    def w(self, off, val): self.m[off:off+4] = struct.pack('<I', val & 0xFFFFFFFF)
    def r(self, off):      return struct.unpack('<I', self.m[off:off+4])[0]

    @staticmethod
    def finc(f): return int(round(f / FS_CLK * (1 << 32))) & 0xFFFFFFFF

    def setup(self, f_lo, dec_log2, fir_bypass=False, src=0):
        self.w(R_CTRL, 0)
        self.w(R_LO_FINC,  self.finc(f_lo))
        self.w(R_DEC_LOG2, dec_log2)
        self.w(R_SRC_SEL,  src)
        ctrl = 1 | 32 | (64 if fir_bypass else 0)
        self.w(R_CTRL, ctrl | 2 | 128)
        self.w(R_CTRL, ctrl)

    def disable(self): self.w(R_CTRL, 0)

    # ---- coefficient access -------------------------------------------------
    #  The filter reads this memory throughout its 155-cycle pass, so writing
    #  while it runs mixes two different filters. Disable, write, re-enable.
    def write_coeffs(self, vals):
        for v in vals:
            if not (COEF_MIN <= int(v) <= COEF_MAX):
                raise ValueError(
                    '%d does not fit an 18-bit signed coefficient (%d .. %d). '
                    'Note that 2**17 = 131072, the scaling for gain 1, is ONE '
                    'TOO LARGE and would be stored as %d.'
                    % (v, COEF_MIN, COEF_MAX, COEF_MIN))
        self.disable()
        for a, v in enumerate(vals):
            self.w(R_COEF_ADDR, a)
            self.w(R_COEF_DATA, int(v) & ((1 << COEFW) - 1))

    def read_coeffs(self, n=NHALF):
        out = []
        for a in range(n):
            self.w(R_COEF_ADDR, a)
            out.append(s32(self.r(R_COEF_DATA)))
        return np.array(out, dtype=np.int64)

    def status(self):
        s = self.r(R_STATUS2)
        return dict(mem_overflow=bool(s & 1), filter_overrun=bool(s & 2),
                    filter_bypassed=bool(s & 4))


acq = Acq()
print('mapped at 0x%08X' % ACQ_BASE)


---
## Before anything else — is the chip running the logic you think it is?

**Read this even if you just rebuilt.** Every wrong answer this notebook has produced so far
turned out to be the right answer about the wrong bitstream.

Register `0x48` holds a hash of the Verilog the running bitstream was synthesised from. It is
baked in at synthesis, so it cannot be faked by a fresh file date. `build_project.tcl` writes
the same value into `bitstream_id.txt`; copy that across with the bitstream and this becomes a
straight yes or no.

| It reads | Meaning |
|---|---|
| the same value as `bitstream_id.txt` | the chip is running your current source. Carry on |
| **zero** | the bitstream was not built by `build_project.tcl` — most likely by pressing Generate Bitstream, which does not force resynthesis. It may contain old logic |
| anything else | an old bitstream is loaded on the board. Reload it |


In [ ]:
build_id = acq.r(R_BUILD_ID)
print('build fingerprint in the chip : 0x%08X' % build_id)

expected = None
for cand in ('bitstream_id.txt', '../bitstream_id.txt'):
    if os.path.exists(cand):
        expected = int(open(cand).read().strip().replace('0x', '').replace('0X', ''), 16)
        print('bitstream_id.txt says         : 0x%08X' % expected)
        break

print()
if build_id == 0:
    print('*** THIS BITSTREAM HAS NO FINGERPRINT ***')
    print()
    print('It was not built by build_project.tcl. Pressing Generate Bitstream in')
    print('Vivado does not regenerate build_id.vh and does not force resynthesis,')
    print('so it can produce a brand-new bitstream file containing week-old logic.')
    print('That has now happened twice on this project.')
    print()
    print('In Vivado: close the project, reopen it, then in the Tcl Console:')
    print('    source C:/rp/prj/v0.94/Test_Project/build_project.tcl')
    print('Copy the new bitstream AND bitstream_id.txt to the board, reload, and')
    print('run this cell again. Do not trust anything below until this passes.')
    raise SystemExit('stopping: unidentified bitstream')
elif expected is not None and build_id != expected:
    print('*** THE BOARD IS RUNNING DIFFERENT LOGIC FROM YOUR SOURCE ***')
    print()
    print('An old bitstream is loaded. Copy the current one across and reload it.')
    print('See ../BITSTREAM_WORKFLOW.pdf.')
    raise SystemExit('stopping: stale bitstream')
else:
    print('Fingerprint present. The chip is identified.')
    if expected is None:
        print('(bitstream_id.txt was not found next to this notebook, so this is not')
        print(' a full check - copy it across with the bitstream to make it one.)')


In [ ]:
# ===========================================================================
#  MEASUREMENT
# ===========================================================================
#  Two rules here, and both matter more than they look.
#
#  1. AMPLITUDE IS MEASURED FROM ONE REGISTER, NOT TWO.
#     |I + jQ| needs two reads microseconds apart, and a new sample arrives
#     every 8 microseconds, so they often come from different samples. The
#     result is biased low by up to 0.9 dB and jitters by up to 48%, both
#     growing with offset frequency - the same shape as the droop being looked
#     for. The test was measuring itself.
#
#  2. THE READS ARE THEMSELVES A SAMPLING PROCESS, AND THEY ARE SLOW.
#     Python manages roughly a thousand reads a second, so anything above about
#     500 Hz folds back and reads as something slower. A 1366 Hz beat came back
#     as 446 Hz. Any method that counts cycles is therefore limited to a few
#     hundred hertz, and the old two-point beat trick - which deliberately moved
#     the reference by 1220 Hz - was measuring an alias and calling it a fault.
#
#  So the generator's offset is found from the PHASE of I + jQ rather than from
#  counting anything. Phase is immune to both problems, and gives the sign for
#  free. See offset_hz below.
# ===========================================================================
COEFW = 18
RAIL  = (1 << 31) - 1
NOISE_FLOOR = 2.0e5          # generous; the real floor is a few times 1e4


def read_I(n=1500, gap=0.001):
    """Read the I sample alone, repeatedly. Returns the values and the span."""
    v, t0 = [], time.time()
    for _ in range(n):
        v.append(s32(acq.r(R_PEEK_I)))
        if gap:
            time.sleep(gap)
    return np.array(v, dtype=float), time.time() - t0


def read_IQ(n=3000, gap=0.0002):
    """Read I and Q together, with a timestamp for each pair.

    The two reads are microseconds apart, which ruins their MAGNITUDE - but it
    adds the same small constant to every phase, and a constant cancels out of
    the phase DIFFERENCE that offset_hz measures. So this is unsuitable for
    amplitude and perfectly suited to frequency.
    """
    t, i, q = [], [], []
    for _ in range(n):
        t.append(time.time())
        i.append(s32(acq.r(R_PEEK_I)))
        q.append(s32(acq.r(R_PEEK_Q)))
        if gap:
            time.sleep(gap)
    return (np.array(t) - t[0], np.array(i, dtype=float), np.array(q, dtype=float))


def amplitude(v):
    """Size of the signal, whether it is standing still or going round.

        amplitude = sqrt( mean^2 + 2 * variance )

    A constant has no variance, so this gives its value. A sine has no mean, so
    this gives sqrt(2) x its spread, which is its amplitude. No special case at
    zero offset, which matters because zero offset is never exactly zero.
    """
    return float(np.sqrt(v.mean() ** 2 + 2 * v.var()))


def measure(f_lo, dec_log2, bypass, settle=0.3, n=1500, gap=0.001):
    """Returns (amplitude, mean, spread, min, max)."""
    acq.setup(f_lo=f_lo, dec_log2=dec_log2, fir_bypass=bypass)
    time.sleep(settle)
    v, span = read_I(n, gap)
    return amplitude(v), v.mean(), v.std(), v.min(), v.max()


def report(label, f_lo, dec_log2, bypass=False):
    """One measurement, printed as a line. Returns the amplitude."""
    a, m, s, lo, hi = measure(f_lo, dec_log2, bypass)
    print('  %-22s amplitude %13.1f   mean %13.1f   min %13.1f   max %13.1f'
          % (label, a, m, lo, hi))
    return a


def offset_hz(f_lo, dec_log2, bypass=True, settle=0.3, n=3000, gap=0.0002):
    """How far the signal is from the reference tone, SIGN INCLUDED.

    The downconverted signal is a vector going round at the difference
    frequency. Its angle therefore advances by a constant amount per second,
    and that rate IS the difference - positive one way round, negative the
    other. Fit a straight line to the angle against time and read off the slope.

    Why this beats counting cycles:
      * it gives the sign, which counting never can
      * it does not care that I and Q are read microseconds apart, because a
        fixed lag adds the same constant to every angle and cancels in the slope
      * how well the straight line fits tells you whether this is a tone at all.
        A real tone gives a residual of a few hundredths of a radian. Noise has
        no coherent phase and gives something near a radian.

    Returns (offset in Hz, residual in radians, samples per second achieved).
    The one limit is that the angle must not advance more than half a turn
    between reads, which caps it at about half the read rate.
    """
    acq.setup(f_lo=f_lo, dec_log2=dec_log2, fir_bypass=bypass)
    time.sleep(settle)
    t, i, q = read_IQ(n, gap)
    rate = (len(t) - 1) / max(t[-1], 1e-9)

    ph = np.unwrap(np.angle(i + 1j * q))
    slope, icept = np.polyfit(t, ph, 1)          # radians per second
    resid = float(np.std(ph - (slope * t + icept)))
    return slope / (2 * np.pi), resid, rate


# ---------------------------------------------------------------------------
#  IS THERE A SIGNAL, AND WHERE EXACTLY IS IT?
#
#  Two failures look identical from inside a test: nothing plugged in, and a
#  frequency typed in wrongly. The window is only +-24 kHz out of a 62 MHz
#  input range, so being 9 MHz out removes the signal as completely as pulling
#  the cable. Both must be caught, and told apart.
# ---------------------------------------------------------------------------
def require_signal(f_nom, dec_log2, search=True):
    """Refuse to continue without a signal. Returns its true frequency."""
    a, _, _, _, _ = measure(f_nom, dec_log2, True, n=800)
    print('  amplitude                     : %14.1f counts' % a)

    if a < NOISE_FLOOR:
        print('')
        print('*** NO SIGNAL ***')
        print('')
        print('  %.0f counts is the noise floor. A connected generator gives millions.' % a)
        print('  Check, in this order:')
        print('    1. the generator OUTPUT is switched on - not just the generator')
        print('    2. the cable is in IN1, not IN2 or OUT1')
        print('    3. the level is a few hundred mV, jumper on LV')
        print('    4. swap the cable')
        if search:
            print('')
            print('  Sweeping the whole input range before giving up:')
            find_generator()
        raise SystemExit('stopping: nothing on the input')

    d, resid, rate = offset_hz(f_nom, dec_log2)
    print('  reads per second              : %14.0f   (this caps what can be seen)' % rate)
    print('  phase fit residual            : %14.3f rad (a tone gives << 1)' % resid)
    print('  offset from what you typed    : %+14.1f Hz  (%+.2f ppm)'
          % (d, 1e6 * d / f_nom))

    if resid > 0.5:
        print('')
        print('*** THERE IS SOMETHING THERE, BUT IT IS NOT A STEADY TONE ***')
        print('')
        print('  The signal is strong enough, but its phase does not advance at a')
        print('  constant rate, so it is not a single frequency. Either the offset')
        print('  is faster than %.0f Hz - more than half the read rate, which folds' % (rate / 2))
        print('  and cannot be untangled - or what is in the window is noise.')
        if search:
            print('')
            print('  Sweeping the whole input range:')
            find_generator()
        raise SystemExit('stopping: no steady tone in the window')

    #  Confirm it, by tuning to where we think it is and checking what is left.
    f_true = f_nom + d
    d2, resid2, _ = offset_hz(f_true, dec_log2)
    print('  after correcting, offset is   : %+14.1f Hz  (should be near zero)' % d2)
    if abs(d2) > max(20.0, 0.1 * abs(d)):
        print('  ...which it is not. The generator is drifting while we measure.')
    print('')
    print('  Signal present at %.4f MHz.' % (f_true / 1e6))
    return f_true


def find_generator(fmin=0.2e6, fmax=60.0e6, quiet=False):
    """Sweep the reference tone across the input range and report what is there.

    Coarse to fine: a small decimation gives a wide window, so few steps cover
    everything; then narrow down. About twenty seconds for the whole range.
    """
    if not quiet:
        print('    searching %.1f - %.1f MHz ...' % (fmin / 1e6, fmax / 1e6))

    f_o = FS_CLK / (1 << 6)
    best_a, best_f = 0.0, None
    f = fmin
    while f < fmax:
        a, _, _, _, _ = measure(f, 6, True, settle=0.02, n=250, gap=0)
        if a > best_a:
            best_a, best_f = a, f
        f += 0.25 * f_o

    if best_f is None or best_a < NOISE_FLOOR:
        if not quiet:
            print('    nothing anywhere in that range (largest %.0f counts).' % best_a)
        return None
    if not quiet:
        print('    something near %.3f MHz (%.0f counts)' % (best_f / 1e6, best_a))

    for dec, span, nstep in ((8, 0.8e6, 17), (10, 0.10e6, 21)):
        vals, cand = [], np.linspace(best_f - span, best_f + span, nstep)
        for f in cand:
            a, _, _, _, _ = measure(f, dec, True, settle=0.05, n=400, gap=0)
            vals.append(a)
        best_f = cand[int(np.argmax(vals))]
        if not quiet:
            print('    refined at R = %-5d : %.4f MHz (%.0f counts)'
                  % (1 << dec, best_f / 1e6, max(vals)))

    d, resid, _ = offset_hz(best_f, 10)
    if resid < 0.5:
        best_f = best_f + d

    if not quiet:
        print('')
        print('    >>> The generator is at %.4f MHz.' % (best_f / 1e6))
        print('        Put this in the setup cell above and run again:')
        print('')
        print('            F_SIG = %.0f' % best_f)
        print('')
    return best_f


In [ ]:
# ===========================================================================
#  THE ONE LINE YOU MUST EDIT
#
#  F_SIG is what your signal generator is set to. The measurement window is only
#  +-24 kHz wide out of a 62 MHz input range, so a wrong value here does not
#  make the signal weak - it makes it VANISH, exactly as if the cable were
#  unplugged. Check it before trusting anything downstream.
#
#  If it is wrong, the test says so and sweeps the whole range to tell you the
#  right value. It does not guess silently.
# ===========================================================================
F_SIG = 10.0e6         # <-- YOUR SIGNAL GENERATOR FREQUENCY [Hz]
DEC   = 10             # log2 of the decimation. 10 -> R = 1024

R     = 1 << DEC
f_out = FS_CLK / R

print('signal generator : %.4f MHz' % (F_SIG / 1e6))
print('decimation       : R = %d,  output rate %.2f kHz' % (R, f_out / 1e3))
print('')

# What the chip is holding right now. If the board has been power cycled since
# anything last wrote here, these are the coefficients built into the bitstream.
designed = acq.read_coeffs()
full_now = np.concatenate([designed, designed[-2::-1]])
print('coefficients currently in the chip:')
print('  centre tap %d, sum %d, gain at DC %.6f'
      % (designed[NHALF-1], full_now.sum(), full_now.sum() / (1 << COEF_FRAC)))

ref_path = 'fir_coeffs.txt'
have_ref = os.path.exists(ref_path)
ref = None
if have_ref:
    ref = np.loadtxt(ref_path).astype(np.int64)
    print('  identical to %s : %s' % (ref_path, np.array_equal(designed, ref[:NHALF])))
else:
    print('  (%s not found next to this notebook - Test D will be skipped)' % ref_path)


---
## Baseline — is there a signal, and how fast is it rotating?

Two questions, both needed before any test below means anything.

**Is there a signal at all?** With nothing connected, the decimator still puts out the noise
floor, a few tens of thousands of counts. A real signal is thousands of times bigger. Tests B
to D compare the filter's output against this baseline, so if the baseline is noise the
comparisons are meaningless.

**How fast is it rotating?** The generator and the board run off separate crystals, so the
reference tone never lands exactly on the signal. The residual beat is measured here, in hertz.
A few tens of hertz at 10 MHz is entirely normal and nothing to fix — the tests are built to
tolerate it.


In [ ]:
# Work out where the generator actually is, before measuring anything with it.
print('Locating the generator (it is not locked to the board):')
F_TRUE = require_signal(F_SIG, DEC)
print()
print('correction BYPASSED, reference tone on the TRUE signal frequency:')
acq.setup(f_lo=F_TRUE, dec_log2=DEC, fir_bypass=True)
time.sleep(0.3)
v, span = read_I(1500)
base_a = amplitude(v)
bt, resid, rate = offset_hz(F_TRUE, DEC)

print('  amplitude        : %14.1f  counts' % base_a)
print('  mean             : %14.1f' % v.mean())
print('  min / max        : %14.1f / %.1f' % (v.min(), v.max()))
print('  measured over    : %14.3f  s' % span)
print()
print('  residual offset  : %+14.1f  Hz   (should be near zero now)' % bt)
print('  phase fit residual:%14.3f  rad  (a tone gives << 1)' % resid)
print()

NOISE_FLOOR = 1.0e5          # generous: the real floor is a few times 1e4
signal_ok   = base_a > 20 * NOISE_FLOOR
rail_ok     = base_a < 0.8 * RAIL

print('  signal well above the noise floor : %s' % signal_ok)
print('  signal well below full scale      : %s' % rail_ok)
print()
if signal_ok and rail_ok:
    print('  BASELINE OK. Use %.1f as the reference amplitude below.' % base_a)
    if abs(bt) > 50:
        print('  Residual offset %.1f Hz is larger than expected - the generator is' % bt)
        print('  drifting while you measure. Results will be a little noisier.')
elif not signal_ok:
    print('  NO SIGNAL. The amplitude is at the noise floor. Check the generator is')
    print('  switched on, its output is enabled, and the cable is in IN1. Nothing')
    print('  below will mean anything until this reads millions of counts.')
else:
    print('  SIGNAL TOO BIG - it is close to full scale and the filter has up to')
    print('  1.3x gain, so it would clip. Turn the generator down.')


---
## Test A — every coefficient zero

A filter of all zeros must produce **exactly zero**, whatever the input.

This tests everything that has nothing to do with the coefficients: that the accumulator is
cleared at the start of each pass, that the output is latched once the pass has finished and
not partway through, and that the strobe announcing a new sample arrives at the right moment.


In [ ]:
acq.write_coeffs(np.zeros(NHALF, dtype=np.int64))
print('wrote zeros;  read back all zero =', bool(np.all(acq.read_coeffs() == 0)))
print()
a, m, s, lo, hi = measure(F_SIG, DEC, bypass=False)
print('  all-zero filter        amplitude %13.1f   min %13.1f   max %13.1f' % (a, lo, hi))
print()
okA = (a == 0.0)
print('TEST A:', 'PASS' if okA else 'FAIL')
if not okA:
    print('   A filter of zeros produced a non-zero output. The accumulator is not')
    print('   being cleared between passes, or the result is latched before the pass')
    print('   has finished. Stop here and report these numbers.')


---
## Test B — the centre tap only

Load one coefficient: the middle one, set to **65536**, which is gain ½. Everything else zero.

That makes the filter a plain wire with the volume turned down by half. So the output amplitude
must be exactly half the baseline.

This tests the multiply, the accumulate, the shift that undoes the coefficient scaling, and the
saturation logic — with an answer we know.

**The centre tap is the one tap that does not use the pre-add**, because it has no partner to
pair with. That is what makes B and C together so informative: B exercises everything *except*
the pre-add, C adds the pre-add and nothing else.


In [ ]:
c = np.zeros(NHALF, dtype=np.int64)
c[NHALF-1] = HALF                    # coef[154] is the centre tap of the 309
acq.write_coeffs(c)
back = acq.read_coeffs()
print('wrote centre tap = %d;  read back %d, rest all zero = %s'
      % (HALF, back[NHALF-1], bool(np.all(back[:NHALF-1] == 0))))
print()
base = report('baseline (bypassed)', F_TRUE, DEC, bypass=True)
got  = report('centre tap only',     F_TRUE, DEC, bypass=False)
gain = got / max(base, 1e-9)
print()
print('  gain    : %.6f   (must be 0.500000)' % gain)
print('  error   : %+.3f %%' % (100 * (gain / 0.5 - 1)))
print()
okB = abs(gain - 0.5) < 0.005
print('TEST B:', 'PASS' if okB else 'FAIL')
if not okB:
    print('   Expected half the input and got %.4f of it.' % gain)
    print('   A clean power of two points at the output shift (COEF_FRAC).')
    print('   A gain near 1.0 means the coefficient did not land where you think.')


---
## Test C — the first tap only

Load coefficient 0 and nothing else, again at 65536. Because the filter stores only its unique
half and mirrors it, that one number becomes **two** taps: the newest sample and the one 308
samples older, each with weight ½.

The result has a shape we can write down:

> gain = |cos(π × 308 × u)|,  where u is the offset as a fraction of the output sample rate

At zero offset that is a gain of **1**. Move a little off and it swings between 1 and 0 — 308
samples is a long lever, so a small change in frequency turns the pair from adding to
cancelling. One full cycle takes u = 1/308 = 0.00325, which at R = 1024 is 397 Hz.

**This is the sharpest test in the notebook.** It is the only one that uses the pre-add, and
the only one that checks the sample history hands back the right samples the right distance
apart. The offsets are set relative to the *measured* beat, so the generator's drift does not
matter.

**If the output rails at ±2147483647:** the pre-add is producing nonsense. That is what a
signed number zero-extended instead of sign-extended looks like — an error of 2³², which is
about a hundred thousand times a normal sample.


In [ ]:
c = np.zeros(NHALF, dtype=np.int64)
c[0] = HALF
acq.write_coeffs(c)
print('wrote first tap = %d' % HALF)
print()

# F_TRUE already accounts for the generator, sign included, so the offsets below
# are the ones actually applied. Two taps 308 samples apart go round a full cycle
# in a very small frequency step - so getting the SIGN of a 150 Hz generator
# error wrong is on its own enough to scramble this whole test.
print('  one full comb cycle = %.5f in u = %.0f Hz'
      % (1.0 / (NTAPS - 1), f_out / (NTAPS - 1)))
print()
print('  %10s %14s %12s %12s %10s' % ('u', 'measured', 'predicted', 'error', 'railing?'))
print('  ' + '-' * 62)

errs, railed, gains, us = [], False, [], []
base_ref, _, _, _, _ = measure(F_TRUE, DEC, bypass=True)
for u in (0.0, 0.0008, 0.0016, 0.0024, 0.0033, 0.0041, 0.0049):
    f_lo = F_TRUE - u * f_out
    aon,  _, _, lo, hi = measure(f_lo, DEC, bypass=False)
    aoff, _, _, _,  _ = measure(f_lo, DEC, bypass=True)
    got  = aon / max(aoff, 1e-9)
    want = abs(np.cos(np.pi * (NTAPS - 1) * u))
    rail = (lo <= -RAIL) or (hi >= RAIL)
    railed = railed or rail
    errs.append(abs(got - want)); gains.append(got); us.append(u)
    print('  %10.4f %14.4f %12.4f %12.4f %10s'
          % (u, got, want, got - want, 'YES' if rail else 'no'))

print()
# ---- recognise the one failure mode we have seen, by its arithmetic --------
#  With two taps of 0.5 the gain can never exceed 1. If instead the output sits
#  at 2^31 / (input amplitude), it is not a gain at all: the output is pinned at
#  the 32-bit limit. That happens when the two samples are added as UNSIGNED
#  numbers - one negative 32-bit sample zero-extended instead of sign-extended
#  leaves 2^32 behind, and 2^32 x 65536 / 2^17 is exactly 2^31.
pinned = (1 << 31) / max(base_ref, 1e-9)
near   = [g for g in gains if abs(g - pinned) < 0.15 * pinned]
if len(near) >= 2:
    print('  >>> %d of %d points sit on 2^31 / baseline = %.2f.' % (len(near), len(gains), pinned))
    print('      That is the 32-bit limit, not a filter gain. The pre-add is being')
    print('      done UNSIGNED - a fault that was found and fixed in')
    print('      red_pitaya_acq_fir.v. If you see this, the chip is running the OLD')
    print('      logic: check the build fingerprint above and reload the bitstream.')
    print()

# ---- backstop: fit a single frequency offset -------------------------------
#  If the response is the right SHAPE but sitting in the wrong place along the
#  frequency axis, then the taps ARE 308 apart and the fault is in knowing where
#  the generator is - not in the hardware. Fitting one free parameter separates
#  those two possibilities cleanly, and the number it returns is the generator
#  error in hertz.
uu = np.array(us); gg = np.array(gains)
cs = np.linspace(-1.0 / (NTAPS - 1), 1.0 / (NTAPS - 1), 4001)
res = [np.sqrt(np.mean((np.abs(np.cos(np.pi * (NTAPS - 1) * (uu + c))) - gg) ** 2))
       for c in cs]
c_best = cs[int(np.argmin(res))]
print('  shape fit: residual %.4f after allowing a frequency offset of %+.1f Hz'
      % (min(res), c_best * f_out))
if min(res) < 0.05:
    print('  >>> The SHAPE matches the theory. The two taps ARE %d samples apart'
          % (NTAPS - 1))
    print('      and the pre-add is sound. Any error in the table above is in')
    print('      knowing exactly where the generator sits, not in the filter.')
print()

okC = (min(res) < 0.05) and not railed
print('TEST C:', 'PASS' if okC else 'FAIL')
if railed:
    print('   THE OUTPUT IS RAILING. With two taps of 0.5 the gain can never exceed 1,')
    print('   so this is not a filter shape at all - the arithmetic is overflowing.')
    print('   The known cause is an unsigned pre-add: see red_pitaya_acq_fir.v,')
    print('   stage 2, and run  python3 validate_fir.py  which now checks for it.')
elif max(errs) >= 0.10:
    print('   The gains are finite but wrong, so the two taps are not %d samples'
          % (NTAPS - 1))
    print('   apart. Check addr_a and addr_b in red_pitaya_acq_fir.v.')


---
## Test D — the designed filter

Put the designed coefficients back and measure the response the whole chain was built for.
With A, B and C passing, this should simply work.


In [ ]:
if not have_ref:
    raise SystemExit('fir_coeffs.txt is not next to this notebook - copy it across first')

half = ref[:NHALF]
acq.write_coeffs(half)
bad = np.nonzero(acq.read_coeffs() != half)[0]
print('wrote the designed filter;  %d of %d read back wrong' % (len(bad), NHALF))
print()

n    = np.arange(NTAPS)
full = np.concatenate([half, half[-2::-1]]).astype(float)

print('  %10s %14s %12s %12s %10s' % ('u', 'measured', 'predicted', 'error', 'railing?'))
print('  ' + '-' * 62)
worst, railed = 0.0, False
for u in (0.0, 0.05, 0.10, 0.15, 0.20):
    f_lo = F_TRUE - u * f_out
    aon,  _, _, lo, hi = measure(f_lo, DEC, bypass=False)
    aoff, _, _, _,  _ = measure(f_lo, DEC, bypass=True)
    got  = aon / max(aoff, 1e-9)
    want = abs(full.dot(np.exp(-2j * np.pi * u * n))) / (1 << COEF_FRAC)
    rail = (lo <= -RAIL) or (hi >= RAIL)
    railed = railed or rail
    worst = max(worst, abs(got - want))
    print('  %10.3f %14.4f %12.4f %12.4f %10s'
          % (u, got, want, got - want, 'YES' if rail else 'no'))

# How precise is one of these numbers? Each reading estimates an amplitude from
# the spread of N asynchronous samples of a sine wave, so it carries a
# statistical error of roughly 1/(2*sqrt(2N)); a RATIO of two carries sqrt(2)
# times that. The pass threshold must respect that, or it is testing how many
# samples were taken rather than testing the hardware.
#
#   300 reads  -> +-2.9% on a ratio  = 0.035 on a gain of 1.2
#  1500 reads  -> +-1.3%             = 0.016
#
# A single point 0.057 out at 300 reads is only 1.6 sigma - noise, not a fault.
# Hence the read count below, and a tolerance derived from it rather than guessed.
N_READS = 1500
prec = np.sqrt(2) / (2 * np.sqrt(2 * N_READS))
tol  = max(3 * prec * 1.3, 0.03)
print()
print('  worst error : %.4f' % worst)
print('  1 sigma on a ratio at %d reads : %.4f' % (N_READS, prec * 1.3))
print('  tolerance (3 sigma)            : %.4f' % tol)
okD = worst < tol and not railed
print('TEST D:', 'PASS' if okD else 'FAIL')
if not okD and worst < 2 * tol:
    print('   Only just outside, and at this level the measurement is the limit,')
    print('   not the filter. Raise n in measure() and repeat before suspecting')
    print('   the hardware.')
if okD:
    print('   The corrector does what it was designed to do. Go back to Test Stage 5,')
    print('   which measures the same thing across a full sweep.')


---
## Restore

Software-written coefficients survive until the board is power cycled or the bitstream is
reloaded, at which point the ones built into the bitstream come back. This leaves the designed
filter loaded and the chain running.


In [ ]:
acq.write_coeffs(ref[:NHALF])
acq.setup(f_lo=F_TRUE, dec_log2=DEC, fir_bypass=False)
time.sleep(0.2)
st = acq.status()
print('designed filter reloaded, chain running')
print('status: overflow=%s  filter overrun=%s  filter bypassed=%s'
      % (st['mem_overflow'], st['filter_overrun'], st['filter_bypassed']))
